# Library 05: network measures

We use the same measures with the same settings for Test, ODI and T20I, so the three networks can be
compared. The reasons behind each choice are written up in `docs/methods_decisions.md`.

**How we treat the edge weights.** A weight here is a *tie strength*: more runs means a stronger tie. Measures
built on shortest paths (geodesics) would read a big number as a long distance, which is the opposite of what
we mean. So:

| Uses the weights (runs) | Treats every tie as one step |
|---|---|
| strength, eigenvector centrality, Louvain communities | betweenness, closeness, geodesic distance, diameter, compactness, clustering, redundancy, k-cores |

Closeness and eigenvector centrality are computed on the main component. All three of our networks turned out
to be a single component, so in practice this is the whole network.

Needs `lib_00_settings`.

In [ ]:
def largest_component(partnership_graph: nx.Graph) -> nx.Graph:
    """The main component: the biggest group of nodes that can all reach each other."""
    biggest_node_set = max(nx.connected_components(partnership_graph), key=len)
    return partnership_graph.subgraph(biggest_node_set).copy()


def louvain_communities(partnership_graph: nx.Graph, seed: int = RANDOM_SEED) -> list[set[str]]:
    """Weighted Louvain communities, largest first.

    Louvain gives the communities back in no fixed order. We sort them by size (and by the smallest player ID
    when two have the same size), so "community 1" always means the same group.
    """
    communities = nx.community.louvain_communities(partnership_graph, weight="weight",
                                                   resolution=LOUVAIN_RESOLUTION, seed=seed)
    return sorted(communities, key=lambda community: (-len(community), min(community)))


def degree_centralisation(partnership_graph: nx.Graph) -> float:
    """Freeman degree centralisation: 0 if every batter has the same degree, 1 for a perfect star."""
    node_count = partnership_graph.number_of_nodes()
    # Edge case: with fewer than 3 nodes the formula divides by zero, so there is no sensible value.
    if node_count < 3:
        return float("nan")
    degrees = [degree for _, degree in partnership_graph.degree()]
    return sum(max(degrees) - degree for degree in degrees) / ((node_count - 1) * (node_count - 2))


def redundancy_of_nodes(partnership_graph: nx.Graph) -> dict[str, float]:
    """Redundancy as defined in the course: the average number of ties a partner has to the node's other partners.

    If a batter has d partners and t of the pairs among them also batted together, each tie is counted from
    both of its ends, so the redundancy is 2t / d. It goes from 0 (no partner knows another) to d - 1 (every
    partner batted with every other partner).
    Edge case: a batter with no partners would mean dividing by zero; we give 0 (it never happens in our data,
    because a node only exists if it has a partner).
    """
    closed_pairs = nx.triangles(partnership_graph)   # t for every node
    return {player: (2 * closed_pairs[player] / degree if degree else 0.0)
            for player, degree in partnership_graph.degree()}


def connectedness_and_compactness(partnership_graph: nx.Graph) -> tuple[float, float]:
    """Connectedness and compactness, the two cohesion measures from the course.

    connectedness = share of pairs of nodes that can reach each other at all
    compactness   = average of 1 / geodesic distance over all pairs (a pair with no path counts as 0)
    Both are 1 for a network where everyone is directly tied to everyone.
    """
    node_count = partnership_graph.number_of_nodes()
    reachable_pairs, inverse_distance_sum = 0, 0.0
    for source, distances in nx.all_pairs_shortest_path_length(partnership_graph):
        for target, distance in distances.items():
            if target != source:
                reachable_pairs += 1
                inverse_distance_sum += 1 / distance
    possible_pairs = node_count * (node_count - 1)
    return reachable_pairs / possible_pairs, inverse_distance_sum / possible_pairs

## Small-world comparison

To say whether a network is "small-world" we need something to compare it with. We generate random graphs with
exactly the same number of nodes and edges, and compare two numbers:

* **C**, the average local clustering coefficient
* **L**, the mean geodesic distance

`sigma = (C / C_random) / (L / L_random)`. A value above 1 means our network is more clustered than chance
while its distances stay about as short as in a random graph, which is what the small-world effect describes.

In [ ]:
def small_world_comparison(partnership_graph: nx.Graph, number_of_graphs: int = NUMBER_OF_RANDOM_GRAPHS,
                           seed: int = RANDOM_SEED) -> dict[str, float]:
    """Clustering and mean distance of the network compared with seeded random graphs of the same size."""
    main_component = largest_component(partnership_graph)
    node_count, edge_count = main_component.number_of_nodes(), main_component.number_of_edges()
    observed_clustering = nx.average_clustering(main_component)
    observed_distance = nx.average_shortest_path_length(main_component)

    # One random generator with our seed hands out a different seed to each random graph. That way the 200
    # graphs are all different, but the same 200 come out on every run.
    random_generator = np.random.default_rng(seed)
    random_clusterings, random_distances = [], []
    for _ in range(number_of_graphs):
        random_graph = nx.gnm_random_graph(node_count, edge_count, seed=int(random_generator.integers(0, 2**31 - 1)))
        # Edge case: a random graph can come out in pieces, and distances are not defined between pieces.
        # We measure it on its main component, the same way we treat our own networks.
        random_graph = largest_component(random_graph)
        random_clusterings.append(nx.average_clustering(random_graph))
        random_distances.append(nx.average_shortest_path_length(random_graph))
    mean_random_clustering = float(np.mean(random_clusterings))
    mean_random_distance = float(np.mean(random_distances))
    return {
        "C_random": mean_random_clustering,
        "L_random": mean_random_distance,
        "C_ratio": observed_clustering / mean_random_clustering,
        "L_ratio": observed_distance / mean_random_distance,
        "small_world_sigma": (observed_clustering / mean_random_clustering) / (observed_distance / mean_random_distance),
    }


def louvain_stability(partnership_graph: nx.Graph, number_of_runs: int = LOUVAIN_STABILITY_RUNS) -> dict[str, float]:
    """Run Louvain with seeds 0 to number_of_runs - 1 and record the range of modularity and community counts."""
    modularity_values, community_counts = [], []
    for seed in range(number_of_runs):
        communities = louvain_communities(partnership_graph, seed=seed)
        modularity_values.append(nx.community.modularity(partnership_graph, communities, weight="weight"))
        community_counts.append(len(communities))
    return {"louvain_Q_min": min(modularity_values), "louvain_Q_max": max(modularity_values),
            "louvain_k_min": min(community_counts), "louvain_k_max": max(community_counts)}

## Measures for each batter

In [ ]:
def node_measures(partnership_graph: nx.Graph) -> pd.DataFrame:
    """One row per batter with all node-level measures, sorted from highest to lowest strength."""
    main_component = largest_component(partnership_graph)
    node_count = partnership_graph.number_of_nodes()

    degree_of = dict(partnership_graph.degree())                          # number of different partners
    strength_of = dict(partnership_graph.degree(weight="weight"))         # total partnership runs
    betweenness_of = nx.betweenness_centrality(partnership_graph, weight=None, normalized=True)
    closeness_of = nx.closeness_centrality(main_component)                # one step per tie, main component
    # We give eigenvector centrality plenty of iterations and a strict tolerance so it settles properly.
    eigenvector_of = nx.eigenvector_centrality(main_component, weight="weight", max_iter=5000, tol=1e-10)
    # Local clustering: by the course convention a batter with fewer than 2 partners gets 0 (no pairs to check).
    clustering_of = nx.clustering(partnership_graph)
    core_number_of = nx.core_number(partnership_graph)                    # highest k-core the batter is in
    redundancy_of = redundancy_of_nodes(partnership_graph)
    community_of = {player: number for number, community in enumerate(louvain_communities(partnership_graph), start=1)
                    for player in community}

    measure_rows = []
    for player_id, node_attributes in partnership_graph.nodes(data=True):
        degree = degree_of[player_id]
        measure_rows.append({
            "format": node_attributes["format"],
            "player_id": player_id,
            "player_name": node_attributes["player_name"],
            "matches_batted": node_attributes["matches_batted"],
            "innings_batted": node_attributes["innings_batted"],
            "partnership_observations": node_attributes["partnership_observations"],
            "first_date": node_attributes.get("first_date", ""),
            "last_date": node_attributes.get("last_date", ""),
            "median_entry_partnership": node_attributes.get("median_entry_partnership", float("nan")),
            "degree": degree,
            "degree_centrality": degree / (node_count - 1),
            "strength": int(strength_of[player_id]),
            "strength_per_innings": strength_of[player_id] / node_attributes["innings_batted"],
            "betweenness": betweenness_of[player_id],
            "in_main_component": player_id in main_component,
            # Edge case: a batter outside the main component has no closeness or eigenvector value. We leave
            # an empty cell (NaN) rather than a made-up number.
            "closeness": closeness_of.get(player_id, float("nan")),
            "eigenvector": eigenvector_of.get(player_id, float("nan")),
            "clustering": clustering_of[player_id],
            "redundancy": redundancy_of[player_id],
            "core_number": core_number_of[player_id],
            "community": community_of[player_id],
        })
    # Ties in strength are broken by player ID, so the order of the table never changes between runs.
    return (pd.DataFrame(measure_rows).sort_values(["strength", "player_id"], ascending=[False, True])
            .reset_index(drop=True))

## Measures for the whole network

In [ ]:
def network_summary(partnership_graph: nx.Graph, node_measure_table: pd.DataFrame) -> dict[str, float]:
    """All network-level measures for one format."""
    node_count, edge_count = partnership_graph.number_of_nodes(), partnership_graph.number_of_edges()
    component_sizes = sorted((len(component) for component in nx.connected_components(partnership_graph)), reverse=True)
    main_component = largest_component(partnership_graph)
    degrees = [degree for _, degree in partnership_graph.degree()]
    strengths = [strength for _, strength in partnership_graph.degree(weight="weight")]
    edge_weights = [attributes["weight"] for _, _, attributes in partnership_graph.edges(data=True)]
    partnerships_per_edge = [attributes["partnership_count"] for _, _, attributes in partnership_graph.edges(data=True)]
    communities = louvain_communities(partnership_graph)
    core_number_of = nx.core_number(partnership_graph)
    highest_core = max(core_number_of.values())
    top_ten_percent_edges = max(1, round(0.1 * edge_count))
    connectedness, compactness = connectedness_and_compactness(partnership_graph)

    summary = {
        "nodes": node_count, "edges": edge_count,
        # Cohesion
        "density": nx.density(partnership_graph),              # ties / possible ties, n(n-1)/2, no self-loops
        "degree_mean": float(np.mean(degrees)),
        "components": len(component_sizes),
        "main_component_nodes": component_sizes[0],
        "component_ratio": (len(component_sizes) - 1) / (node_count - 1),
        "connectedness": connectedness,
        "compactness": compactness,
        # Degree distribution and centralisation
        "degree_median": float(np.median(degrees)), "degree_min": min(degrees), "degree_max": max(degrees),
        "degree_sd": float(np.std(degrees, ddof=1)),
        "degree_centralisation": degree_centralisation(partnership_graph),
        # Weights
        "strength_mean": float(np.mean(strengths)), "strength_median": float(np.median(strengths)),
        "strength_max": max(strengths),
        "edge_weight_total": int(sum(edge_weights)), "edge_weight_mean": float(np.mean(edge_weights)),
        "edge_weight_median": float(np.median(edge_weights)), "edge_weight_max": max(edge_weights),
        "edges_single_partnership_share": sum(1 for count in partnerships_per_edge if count == 1) / edge_count,
        "top10pct_edges_weight_share": float(np.sort(edge_weights)[::-1][:top_ten_percent_edges].sum() / sum(edge_weights)),
        # Clustering, distances and redundancy
        "avg_clustering": nx.average_clustering(partnership_graph),
        "transitivity": nx.transitivity(partnership_graph),
        "mean_geodesic_distance": nx.average_shortest_path_length(main_component),
        "diameter": nx.diameter(main_component),
        "mean_redundancy": float(node_measure_table["redundancy"].mean()),
        # Core-periphery
        "max_core_k": highest_core,
        "max_core_nodes": sum(1 for core in core_number_of.values() if core == highest_core),
        # Communities
        "louvain_communities": len(communities),
        "louvain_modularity": nx.community.modularity(partnership_graph, communities, weight="weight"),
        "louvain_sizes": "|".join(str(len(community)) for community in communities),
        "median_innings_batted": statistics.median(node_measure_table["innings_batted"]),
    }
    summary.update(small_world_comparison(partnership_graph))
    summary.update(louvain_stability(partnership_graph))
    return summary